# گام ۳: بخش طراحی (سؤال‌های تحلیلی)
عرفان دهستانی | بازی دوز

توی این بخش من فکر کردم که قبل از رسم نمودار باید بدانیم تو داده‌ها دنبال چی بگردیم. برای همین ۶ تا سؤال نوشتم. زیر هر سؤال نوشتم چرا برایم مهم بود، از کدام ستون‌ها استفاده می‌شود و چه نمودار یا آماری به درد می‌خورد. بعدش با یک کد کوتاه جواب اولیه را از داده درآوردم تا مطمئن شوم سؤال جواب دارد.

داده‌ها:
- فقط `ttt_games.csv`: ۲۰۰ بازی با تعداد گره‌های مینیمکس و آلفا-بتا (همان دیتاست این گام)

In [ ]:
import pandas as pd

g = pd.read_csv("ttt_games.csv", dtype={"start_board": str})            # ۲۰۰ بازی
print("شکل داده:", g.shape)

## سؤال ۱: آلفا-بتا چقدر کمتر از مینیمکس گره می‌بیند؟
دلیل: کل هدف این گام مقایسه‌ی هزینه‌ی دو الگوریتم است.
ستون‌ها: `nodes_minimax`، `nodes_alphabeta`، `opening_moves`
نمودار: میله‌ای (مقیاس لگاریتمی) و جعبه‌ای

In [ ]:
print("آلفا-بتا فقط", round(g.nodes_alphabeta.sum() / g.nodes_minimax.sum() * 100, 1), "درصد گره‌های مینیمکس را می‌بیند")
t = g.groupby("opening_moves")[["nodes_minimax", "nodes_alphabeta"]].mean()
t["نسبت مینیمکس به آلفا-بتا"] = (t.nodes_minimax / t.nodes_alphabeta).round(1)
t.round(0)

## سؤال ۲: وقتی خانه‌ی خالی بیشتر است، جست‌وجو سخت‌تر می‌شود؟
دلیل: فکر کردم هرچه تخته پرتر شود درخت بازی کوچک‌تر می‌شود و می‌خواستم ببینم داده همین را نشان می‌دهد یا نه.
ستون‌ها: `empty_cells`، `nodes_minimax`، `nodes_alphabeta`
نمودار: پراکندگی با محور لگاریتمی، به‌علاوه ضریب همبستگی

In [ ]:
print(g[["empty_cells", "nodes_minimax", "nodes_alphabeta", "moves"]].corr().round(2))

## سؤال ۳: شروع‌ها چند درصد برای X برد، مساوی یا باخت‌اند؟
دلیل: `start_value` نتیجه‌ی بازی کامل از دید X است (۱ برد X، ۰ مساوی، ۱- برد O).
نمودار: میله‌ای

In [ ]:
print(g.start_value.value_counts(normalize=True).round(2))
print(g.groupby("opening_moves").start_value.apply(lambda v: (v == 1).mean()).round(2))

## سؤال ۴: X کامل از موقعیت بازنده هم می‌تواند ببرد؟
دلیل: چون O تصادفی بازی می‌کند و اشتباه می‌کند، می‌خواستم ببینم عامل ما چقدر از نظریه بهتر عمل می‌کند.
ستون‌ها: `start_value`، `winner`
نمودار: میله‌ی انباشته

In [ ]:
pd.crosstab(g.start_value, g.winner)

## سؤال ۵: بازی‌هایی که X می‌برد کوتاه‌ترند یا مساوی‌ها؟
دلیل: می‌خواستم ببینم نتیجه‌ی بازی با تعداد حرکت‌ها رابطه دارد یا نه.
ستون‌ها: `moves`، `winner`
نمودار: جعبه‌ای

In [ ]:
print(g.groupby("winner").moves.mean().round(2))

## سؤال ۶: وقتی بازی با نوبت O شروع می‌شود، X هنوز می‌برد؟
دلیل: وقتی تعداد حرکت‌های اول فرد است، نوبت O است و می‌خواستم ببینم این X را ضعیف می‌کند یا نه.
ستون‌ها: `to_move`، `winner`
نمودار: میله‌ی انباشته

In [ ]:
pd.crosstab(g.to_move, g.winner, normalize="index").round(2)

## جمع‌بندی
- آلفا-بتا حدود ۳٫۶٪ گره‌های مینیمکس را می‌بیند، چون شاخه‌های بی‌فایده را هرس می‌کند. پس برای دوز انتخاب خوبی است.
- هرچه خانه‌ی خالی بیشتر باشد، هزینه‌ی جست‌وجو بیشتر است (همبستگی حدود ۰٫۸).
- X کامل از ۱۴ شروع بازنده، ۱۱ بار برد، چون O تصادفی اشتباه می‌کند.
- بازی‌هایی که با برد X تمام می‌شوند کوتاه‌ترند و مساوی‌ها طولانی‌ترند.